In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, kruskal, spearmanr
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 160)
DATA = "data"


## 공통 유틸

In [2]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    phi2 = chi2 / n
    r, k = ct.shape
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)
    v = np.sqrt(phi2_corr / max(min(k_corr - 1, r_corr - 1), 1e-9))
    return v, p


def correlation_ratio_kw(categories, measurements):
    """categorical(명목) vs continuous(비율/비중) 관계의 효과크기(eta)와
    유의성(Kruskal-Wallis, 비모수)을 함께 반환.
    ANOVA(F-test) 대신 Kruskal-Wallis를 쓰는 이유:
    방문유형 비중처럼 0~1 사이에 몰려있고 0이 많은(zero-inflated) 데이터는
    정규성·등분산 가정을 만족하기 어렵기 때문."""
    df = pd.DataFrame({"cat": categories, "val": measurements}).dropna()
    groups = [g["val"].values for _, g in df.groupby("cat")]
    grand_mean = df["val"].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((df["val"] - grand_mean) ** 2).sum()
    eta = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0

    valid_groups = [g for g in groups if len(g) >= 2]
    if len(valid_groups) >= 2:
        stat, p_val = kruskal(*valid_groups)
    else:
        stat, p_val = np.nan, np.nan
    return eta, p_val, len(df)


def fdr_flag(pvals, alpha=0.05):
    """Benjamini-Hochberg FDR 보정. reject 여부와 보정된 p-value를 반환."""
    pvals = np.asarray(pvals, dtype=float)
    mask = ~np.isnan(pvals)
    reject = np.full(pvals.shape, False)
    p_adj = np.full(pvals.shape, np.nan)
    if mask.sum() > 0:
        rej, p_corr, _, _ = multipletests(pvals[mask], alpha=alpha, method="fdr_bh")
        reject[mask] = rej
        p_adj[mask] = p_corr
    return reject, p_adj


def effect_label_eta(eta):
    if eta >= 0.5:
        return "큰 효과"
    if eta >= 0.3:
        return "중간 효과"
    if eta >= 0.1:
        return "작은 효과"
    return "무시할 수준"


def effect_label_rho(rho):
    a = abs(rho)
    if a >= 0.5:
        return "큰 효과"
    if a >= 0.3:
        return "중간 효과"
    if a >= 0.1:
        return "작은 효과"
    return "무시할 수준"


## 데이터 로드

In [3]:
master = pd.read_csv(f"{DATA}/tn_traveller_master_여행객 Master_E.csv")
travel = pd.read_csv(f"{DATA}/tn_travel_여행_E.csv")
visit = pd.read_csv(f"{DATA}/tn_visit_area_info_방문지정보_E.csv")
activity = pd.read_csv(f"{DATA}/tn_activity_his_활동내역_E.csv")
companion = pd.read_csv(f"{DATA}/tn_companion_info_동반자정보_E.csv")
codeb = pd.read_csv(f"{DATA}/tc_codeb_코드B.csv")


def code_map(cd_a):
    sub = codeb[codeb["cd_a"] == cd_a][["cd_b", "cd_nm"]]
    return {int(k): v for k, v in sub.values}


tmt_map = code_map("TMT")
mis_map = code_map("MIS")
vis_map = code_map("VIS")


## 동반자 유형 / 목적 파생

In [4]:
def companion_group(rel_codes):
    s = set(rel_codes)
    if s == {1}:
        return "부부/커플"
    if 2 in s:
        return "자녀동반(가족)"
    if 7 in s or 8 in s:
        return "친구/연인"
    if 3 in s or 4 in s:
        return "부모/조부모 동반"
    return "기타/단체"


comp_grp = (
    companion.groupby("TRAVEL_ID")["REL_CD"].apply(companion_group)
    .reset_index(name="COMPANION_TYPE")
)
travel = travel.merge(comp_grp, on="TRAVEL_ID", how="left")
travel["COMPANION_TYPE"] = travel["COMPANION_TYPE"].fillna("나홀로(추정)")

# TRAVEL_PURPOSE 결측/포맷 방어
n_purpose_na = travel["TRAVEL_PURPOSE"].isna().sum()
print(f"[전처리] TRAVEL_PURPOSE 결측: {n_purpose_na}건")
travel["PURPOSE_FIRST"] = (
    travel["TRAVEL_PURPOSE"].dropna().str.split(";").str[0].astype(int)
)
travel["PURPOSE_FIRST"] = travel["PURPOSE_FIRST"].reindex(travel.index)

print("=" * 70)
print("[SECTION 0] 전처리 검증")
print("=" * 70)

# ------------------------------------------------------------
# (1) 복합키 무결성
# ------------------------------------------------------------
assert visit.duplicated(subset=["TRAVEL_ID", "VISIT_AREA_ID"]).sum() == 0, \
    "복합키가 깨졌습니다 - 조인 로직을 재검토하세요"
print("(TRAVEL_ID, VISIT_AREA_ID) 복합키 유일성: OK")

# ------------------------------------------------------------
# (2) 로지스틱 지점 제외
# ------------------------------------------------------------
LOGISTICS_TYPES = [21, 23, 24]
n_before = len(visit)
tourism_visit = visit[~visit["VISIT_AREA_TYPE_CD"].isin(LOGISTICS_TYPES)].copy()
print(f"로지스틱 지점 제외: {n_before}건 -> {len(tourism_visit)}건 "
      f"({(n_before - len(tourism_visit)) / n_before:.1%} 제거)")

# ------------------------------------------------------------
# (3) 체류시간 0분 플래그 + 실제 활용 (민감도 체크)
# ------------------------------------------------------------
tourism_visit["ZERO_STAY_FLAG"] = tourism_visit["RESIDENCE_TIME_MIN"] == 0
zero_rate = tourism_visit["ZERO_STAY_FLAG"].mean()
print(f"체류시간 0분 방문: {tourism_visit['ZERO_STAY_FLAG'].sum()}건 ({zero_rate:.1%})")

zero_by_type = (
    tourism_visit.groupby("VISIT_AREA_TYPE_CD")["ZERO_STAY_FLAG"]
    .mean().sort_values(ascending=False)
)
print("\n-- 방문유형별 0분 방문 비율 (특정 유형에 쏠려있는지 확인) --")
print(zero_by_type.rename(index=lambda c: vis_map.get(c, c)).round(3).to_string())
# 특정 유형에 0분 방문이 몰려있다면(예: 경유지) 그 유형은 별도 해석 주의

# ------------------------------------------------------------
# (4) 이상치 점검 - RESIDENCE_TIME_MIN, DGSTFN
# ------------------------------------------------------------
print("\n-- RESIDENCE_TIME_MIN 분포 (0분 제외) --")
stay_nonzero = tourism_visit.loc[~tourism_visit["ZERO_STAY_FLAG"], "RESIDENCE_TIME_MIN"]
print(stay_nonzero.describe(percentiles=[.01, .25, .5, .75, .99]).round(1).to_string())
q99 = stay_nonzero.quantile(0.99)
n_extreme = (stay_nonzero > q99).sum()
print(f"99백분위({q99:.0f}분) 초과 방문: {n_extreme}건 - 클러스터링 피처(AVG_STAY)에 영향 가능, "
      f"필요시 winsorize 검토")

print("\n-- DGSTFN(만족도) 결측/범위 점검 --")
print(f"결측: {tourism_visit['DGSTFN'].isna().sum()}건, "
      f"범위: {tourism_visit['DGSTFN'].min()}~{tourism_visit['DGSTFN'].max()}")

# ------------------------------------------------------------
# (5) 조인 손실(커버리지) 리포트
# ------------------------------------------------------------
print("\n-- 조인 손실(커버리지) 리포트 --")
v_pre = tourism_visit.merge(
    travel[["TRAVEL_ID", "TRAVELER_ID", "PURPOSE_FIRST", "COMPANION_TYPE"]],
    on="TRAVEL_ID", how="left"
)
n_travel_missing = v_pre["TRAVELER_ID"].isna().sum()
print(f"tourism_visit 중 travel 매칭 안 된 행: {n_travel_missing}건 "
      f"({n_travel_missing / len(v_pre):.1%})")
v = v_pre.dropna(subset=["TRAVELER_ID"]).copy()

visit_share_traveler = pd.crosstab(v["TRAVELER_ID"], v["VISIT_AREA_TYPE_CD"], normalize="index")
visit_share_travel = pd.crosstab(v["TRAVEL_ID"], v["VISIT_AREA_TYPE_CD"], normalize="index")

n_master_travelers = master["TRAVELER_ID"].nunique()
n_matched_travelers = master["TRAVELER_ID"].isin(visit_share_traveler.index).sum()
print(f"master 여행객 {n_master_travelers}명 중 방문기록 매칭: {n_matched_travelers}명 "
      f"({n_matched_travelers / n_master_travelers:.1%}) "
      f"- 나머지 {n_master_travelers - n_matched_travelers}명은 inner join에서 자동 제외됨")

# ------------------------------------------------------------
# (6) 핵심 컬럼 결측치 점검
# ------------------------------------------------------------
print("\n-- 핵심 컬럼 결측치 --")
styl_cols = [f"TRAVEL_STYL_{i}" for i in range(1, 9)]
motive_cols = ["TRAVEL_MOTIVE_1", "TRAVEL_MOTIVE_2", "TRAVEL_MOTIVE_3"]
for c in styl_cols + motive_cols:
    na = master[c].isna().sum()
    if na > 0:
        print(f"  {c}: 결측 {na}건 ({na/len(master):.1%})")
print("(표시 없는 컬럼은 결측 0건)")

print()


[전처리] TRAVEL_PURPOSE 결측: 0건
[SECTION 0] 전처리 검증
(TRAVEL_ID, VISIT_AREA_ID) 복합키 유일성: OK
로지스틱 지점 제외: 21384건 -> 15319건 (28.4% 제거)
체류시간 0분 방문: 1407건 (9.2%)

-- 방문유형별 0분 방문 비율 (특정 유형에 쏠려있는지 확인) --
VISIT_AREA_TYPE_CD
역, 터미널, 고속도로 휴게소                   0.462
상점                                 0.169
기타                                 0.116
역사/유적/종교 시설(문화재, 박물관, 촬영지, 절 등)    0.085
산책로, 둘레길 등                         0.075
상업지구(거리, 시장, 쇼핑시설)                 0.062
친구/친지집                             0.043
식당/카페                              0.034
자연관광지                              0.033
체험 활동 관광지                          0.027
테마시설(놀이공원, 워터파크)                   0.026
지역 축제/행사                           0.024
문화 시설(공연장, 영화관, 전시관 등)             0.017
레저/스포츠 관련 시설(스키, 카트, 수상레저)         0.006

-- RESIDENCE_TIME_MIN 분포 (0분 제외) --
count    13813.0
mean        83.0
std        104.0
min         30.0
1%          30.0
25%         30.0
50%         60.0
75%         90.0
99%        480.0
max       2280.0
99백분위(480

## SECTION 2. STYL(성향) vs 실제 방문유형 비중

In [5]:
print("=" * 70)
print("[SECTION 2] STYL(성향) vs 실제 방문유형 비중 - rho & 유의성(Spearman) + FDR 보정")
print("=" * 70)

joined_styl = master.set_index("TRAVELER_ID")[styl_cols].join(visit_share_traveler, how="inner")

rows = []
for styl in styl_cols:
    for vt in visit_share_traveler.columns:
        rho, p = spearmanr(joined_styl[styl], joined_styl[vt])
        rows.append({"STYL": styl, "VIS_TYPE": vis_map.get(vt, vt), "rho": rho, "p": p})
styl_result = pd.DataFrame(rows)

reject, p_adj = fdr_flag(styl_result["p"].values)
styl_result["p_fdr"] = p_adj
styl_result["sig_fdr"] = reject
styl_result["effect"] = styl_result["rho"].apply(effect_label_rho)

n_raw_sig = (styl_result["p"] < 0.05).sum()
n_fdr_sig = styl_result["sig_fdr"].sum()
print(f"검정 총 {len(styl_result)}쌍")
print(f"  raw p<0.05 (보정 전, 다중비교 문제 있음): {n_raw_sig}건")
print(f"  FDR(BH) 보정 후 유의: {n_fdr_sig}건  <- 이 기준을 신뢰할 것")

sig_after_fdr = styl_result[styl_result["sig_fdr"]].sort_values("rho", key=abs, ascending=False)
print(f"\n-- FDR 보정 후 유의한 전체 {len(sig_after_fdr)}개 항목 (rho 절대값 순) --")
print(sig_after_fdr.head(15).to_string(index=False))

n_meaningful = (sig_after_fdr["effect"] != "무시할 수준").sum()
print(f"\n(참고) 그중 효과크기가 '작은 효과' 이상인 항목: {n_meaningful}개")
print("나머지는 표본이 커서(n이 클수록 작은 차이도 유의하게 나옴) 통계적으로는 유의하지만")
print("실질적 크기는 작다는 뜻 - '유의하지 않다'가 아니라 '통계적 유의성과 실질적 중요성을 구분해서 보라'는 의미")

print()


[SECTION 2] STYL(성향) vs 실제 방문유형 비중 - rho & 유의성(Spearman) + FDR 보정
검정 총 112쌍
  raw p<0.05 (보정 전, 다중비교 문제 있음): 29건
  FDR(BH) 보정 후 유의: 22건  <- 이 기준을 신뢰할 것

-- FDR 보정 후 유의한 전체 22개 항목 (rho 절대값 순) --
         STYL                        VIS_TYPE       rho            p        p_fdr  sig_fdr effect
TRAVEL_STYL_1                역, 터미널, 고속도로 휴게소  0.117649 2.400162e-09 2.688181e-07     True  작은 효과
TRAVEL_STYL_1                           자연관광지 -0.108405 3.878550e-08 2.171988e-06     True  작은 효과
TRAVEL_STYL_2                           식당/카페 -0.088208 7.903137e-06 2.950505e-04     True 무시할 수준
TRAVEL_STYL_4                역, 터미널, 고속도로 휴게소  0.085938 1.346731e-05 3.770846e-04     True 무시할 수준
TRAVEL_STYL_4 역사/유적/종교 시설(문화재, 박물관, 촬영지, 절 등)  0.077703 8.355629e-05 1.871661e-03     True 무시할 수준
TRAVEL_STYL_5      레저/스포츠 관련 시설(스키, 카트, 수상레저)  0.073680 1.916654e-04 3.577754e-03     True 무시할 수준
TRAVEL_STYL_8                테마시설(놀이공원, 워터파크)  0.070319 3.720297e-04 5.952476e-03     True 무시할 수준
TRAVEL_STYL_4         

## SECTION 3. TMT(동기)/MIS(목적) vs 실제 방문유형 비중

In [ ]:

print("=" * 70)
print("[SECTION 3] TMT(동기)/MIS(목적) vs 실제 방문유형 비중 - eta & Kruskal-Wallis + FDR")
print("=" * 70)

tmt = master.set_index("TRAVELER_ID")["TRAVEL_MOTIVE_1"]
joined_tmt = visit_share_traveler.join(tmt, how="inner").dropna(subset=["TRAVEL_MOTIVE_1"])

rows = []
for vt in visit_share_traveler.columns:
    eta, p, n = correlation_ratio_kw(joined_tmt["TRAVEL_MOTIVE_1"], joined_tmt[vt])
    rows.append({"VIS_TYPE": vis_map.get(vt, vt), "eta": eta, "p": p, "n": n})
tmt_result = pd.DataFrame(rows)
reject, p_adj = fdr_flag(tmt_result["p"].values)
tmt_result["p_fdr"] = p_adj
tmt_result["sig_fdr"] = reject
tmt_result["effect"] = tmt_result["eta"].apply(effect_label_eta)
tmt_result = tmt_result.sort_values("eta", ascending=False)
print("-- TMT(1순위 동기), 검정: Kruskal-Wallis (proportion 데이터의 비정규성 반영) --")
print(tmt_result.to_string(index=False))
print(f"raw p<0.05: {(tmt_result['p'] < 0.05).sum()} / {len(tmt_result)}  |  "
      f"FDR 보정 후 유의: {tmt_result['sig_fdr'].sum()} / {len(tmt_result)}")

purpose = travel.set_index("TRAVEL_ID")["PURPOSE_FIRST"]
joined_mis = visit_share_travel.join(purpose, how="inner").dropna(subset=["PURPOSE_FIRST"])
rows = []
for vt in visit_share_travel.columns:
    eta, p, n = correlation_ratio_kw(joined_mis["PURPOSE_FIRST"], joined_mis[vt])
    rows.append({"VIS_TYPE": vis_map.get(vt, vt), "eta": eta, "p": p, "n": n})
mis_result = pd.DataFrame(rows)
reject, p_adj = fdr_flag(mis_result["p"].values)
mis_result["p_fdr"] = p_adj
mis_result["sig_fdr"] = reject
mis_result["effect"] = mis_result["eta"].apply(effect_label_eta)
mis_result = mis_result.sort_values("eta", ascending=False)
print("\n-- MIS(1순위 목적) --")
print(mis_result.to_string(index=False))
print(f"raw p<0.05: {(mis_result['p'] < 0.05).sum()} / {len(mis_result)}  |  "
      f"FDR 보정 후 유의: {mis_result['sig_fdr'].sum()} / {len(mis_result)}")

# ------------------------------------------------------------
# cramers_v 실사용: 범주형(COMPANION_TYPE) x 범주형(PURPOSE_FIRST) 연관성
# ------------------------------------------------------------
print("\n-- (범주형 x 범주형) COMPANION_TYPE vs PURPOSE_FIRST: Cramer's V --")
v_stat, p_cv = cramers_v(travel["COMPANION_TYPE"], travel["PURPOSE_FIRST"])
print(f"Cramer's V = {v_stat:.3f}, p = {p_cv:.4g}")

print()


[SECTION 3] TMT(동기)/MIS(목적) vs 실제 방문유형 비중 - eta & Kruskal-Wallis + FDR
-- TMT(1순위 동기), 검정: Kruskal-Wallis (proportion 데이터의 비정규성 반영) --
                       VIS_TYPE      eta            p    n        p_fdr  sig_fdr effect
역사/유적/종교 시설(문화재, 박물관, 촬영지, 절 등) 0.230666 3.688613e-19 2558 5.164059e-18     True  작은 효과
         문화 시설(공연장, 영화관, 전시관 등) 0.158614 1.188469e-10 2558 8.319284e-10     True  작은 효과
                       지역 축제/행사 0.122279 5.810756e-06 2558 2.711686e-05     True  작은 효과
     레저/스포츠 관련 시설(스키, 카트, 수상레저) 0.120709 2.921533e-05 2558 1.022536e-04     True  작은 효과
               역, 터미널, 고속도로 휴게소 0.115153 1.242882e-03 2558 2.900057e-03     True  작은 효과
               테마시설(놀이공원, 워터파크) 0.112059 3.041486e-04 2558 8.516160e-04     True  작은 효과
                      체험 활동 관광지 0.105858 6.076634e-03 2558 1.063411e-02     True  작은 효과
                             기타 0.098851 1.391449e-02 2558 1.948028e-02     True 무시할 수준
                     산책로, 둘레길 등 0.097588 1.659678e-03 2558 3.319356e-03  

## SECTION 4. TMT 다중동기 클러스터링

In [ ]:
print("=" * 70)
print("[SECTION 4] TMT 다중동기 클러스터링 (동기 유형 도출)")
print("=" * 70)

codes = range(1, 11)
motive_vec = pd.DataFrame(0.0, index=master["TRAVELER_ID"], columns=[f"TMT_{c}" for c in codes])
for rank, col in enumerate(motive_cols):
    weight = 3 - rank
    for tid, code in zip(master["TRAVELER_ID"], master[col]):
        if pd.notna(code):
            motive_vec.loc[tid, f"TMT_{int(code)}"] += weight

X = StandardScaler().fit_transform(motive_vec)
sil_scores = {}
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    sil_scores[k] = silhouette_score(X, km.labels_)
best_k = max(sil_scores, key=sil_scores.get)
print("k별 실루엣 스코어:", {k: round(v, 3) for k, v in sil_scores.items()})
print(f"선택된 k = {best_k} (실루엣={sil_scores[best_k]:.3f})")
if sil_scores[best_k] < 0.25:
    print("  [주의] 실루엣 < 0.25: 군집 간 경계가 뚜렷하지 않음. 해석 시 '탐색적 결과'로 취급할 것")

# 클러스터 안정성 체크: random_state를 바꿔도 k별 순위가 유지되는지
stability_runs = []
for seed in [1, 7, 99]:
    s = {}
    for k in range(3, 9):
        km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(X)
        s[k] = silhouette_score(X, km.labels_)
    stability_runs.append(max(s, key=s.get))
is_stable = len(set(stability_runs + [best_k])) == 1
print(f"안정성 체크: 원 분석(seed=42) k={best_k}, 재실행 3회(seed=1,7,99) = {stability_runs} "
      f"{'-> 일관됨' if is_stable else '-> seed에 따라 다른 k가 나옴 (k=' + str(best_k) + '는 참고용으로 취급)'}")

km = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X)
master["MOTIVE_CLUSTER"] = km.labels_

print("\n-- 클러스터별 대표 동기(top3) --")
centers = pd.DataFrame(km.cluster_centers_, columns=motive_vec.columns)
for i, row in centers.iterrows():
    top3 = row.sort_values(ascending=False).head(3)
    names = [tmt_map[int(c.split("_")[1])] for c in top3.index]
    n_members = (master["MOTIVE_CLUSTER"] == i).sum()
    print(f"클러스터 {i} (n={n_members}): {names}")

print()


[SECTION 4] TMT 다중동기 클러스터링 (동기 유형 도출)


C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Win

k별 실루엣 스코어: {3: 0.192, 4: 0.202, 5: 0.221, 6: 0.266, 7: 0.331, 8: 0.283}
선택된 k = 7 (실루엣=0.331)


C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Win

안정성 체크: 원 분석(seed=42) k=7, 재실행 3회(seed=1,7,99) = [8, 8, 8] -> seed에 따라 다른 k가 나옴 (k=7는 참고용으로 취급)

-- 클러스터별 대표 동기(top3) --
클러스터 0 (n=187): ['운동, 건강 증진 및 충전', '쉴 수 있는 기회, 육체 피로 해결 및 정신적인 휴식', '진정한 자아 찾기 또는 자신을 되돌아볼 기회 찾기']
클러스터 1 (n=1568): ['일상적인 환경 및 역할에서의 탈출, 지루함 탈피', '여행 동반자와의 친밀감 및 유대감 증진', '쉴 수 있는 기회, 육체 피로 해결 및 정신적인 휴식']
클러스터 2 (n=102): ['특별한 목적(칠순여행, 신혼여행, 수학여행, 인센티브여행)', '여행 동반자와의 친밀감 및 유대감 증진', '기타']
클러스터 3 (n=297): ['역사 탐방, 문화적 경험 등 교육적 동기', '새로운 경험 추구', '기타']
클러스터 4 (n=135): ['진정한 자아 찾기 또는 자신을 되돌아볼 기회 찾기', '새로운 경험 추구', '쉴 수 있는 기회, 육체 피로 해결 및 정신적인 휴식']
클러스터 5 (n=49): ['기타', 'SNS 사진 등록 등 과시', '특별한 목적(칠순여행, 신혼여행, 수학여행, 인센티브여행)']
클러스터 6 (n=222): ['SNS 사진 등록 등 과시', '기타', '새로운 경험 추구']



C:\Users\skgp2\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(


## SECTION 4b. 클러스터별 실제 행동 차이 검정 + 사후검정

In [8]:
prof = master.set_index("TRAVELER_ID")["MOTIVE_CLUSTER"].to_frame().join(visit_share_traveler, how="inner")
print("-- 클러스터별 방문유형 비중 차이 (Kruskal-Wallis) + FDR 보정 + 사후검정(Dunn) --")
kw_rows = []
for vt in visit_share_traveler.columns:
    groups = [g[vt].values for _, g in prof.groupby("MOTIVE_CLUSTER")]
    stat, p = kruskal(*groups)
    kw_rows.append({"VIS_TYPE": vis_map.get(vt, vt), "vt_code": vt, "H": stat, "p": p})
kw_df = pd.DataFrame(kw_rows)
reject, p_adj = fdr_flag(kw_df["p"].values)
kw_df["p_fdr"] = p_adj
kw_df["sig_fdr"] = reject

for _, r in kw_df[kw_df["sig_fdr"]].iterrows():
    print(f"VIS_TYPE {r['VIS_TYPE']}: H={r['H']:.2f}, p={r['p']:.4f}, p_fdr={r['p_fdr']:.4f} (FDR 유의)")
    # 사후검정: 어떤 클러스터 쌍이 다른지
    dunn = sp.posthoc_dunn(prof, val_col=r["vt_code"], group_col="MOTIVE_CLUSTER", p_adjust="bonferroni")
    sig_pairs = [(a, b) for a in dunn.index for b in dunn.columns
                 if a < b and dunn.loc[a, b] < 0.05]
    print(f"    유의한 클러스터 쌍(Dunn, bonferroni<0.05): {sig_pairs if sig_pairs else '없음'}")

print()


-- 클러스터별 방문유형 비중 차이 (Kruskal-Wallis) + FDR 보정 + 사후검정(Dunn) --
VIS_TYPE 자연관광지: H=18.43, p=0.0052, p_fdr=0.0122 (FDR 유의)
    유의한 클러스터 쌍(Dunn, bonferroni<0.05): [(0, 3), (1, 3)]
VIS_TYPE 역사/유적/종교 시설(문화재, 박물관, 촬영지, 절 등): H=230.01, p=0.0000, p_fdr=0.0000 (FDR 유의)
    유의한 클러스터 쌍(Dunn, bonferroni<0.05): [(0, 3), (1, 3), (2, 3), (3, 4), (3, 5), (3, 6)]
VIS_TYPE 문화 시설(공연장, 영화관, 전시관 등): H=63.71, p=0.0000, p_fdr=0.0000 (FDR 유의)
    유의한 클러스터 쌍(Dunn, bonferroni<0.05): [(0, 3), (1, 3), (1, 4), (2, 3), (3, 6)]
VIS_TYPE 상업지구(거리, 시장, 쇼핑시설): H=35.44, p=0.0000, p_fdr=0.0000 (FDR 유의)
    유의한 클러스터 쌍(Dunn, bonferroni<0.05): [(0, 1), (0, 4), (0, 5), (1, 3), (3, 5)]
VIS_TYPE 레저/스포츠 관련 시설(스키, 카트, 수상레저): H=57.78, p=0.0000, p_fdr=0.0000 (FDR 유의)
    유의한 클러스터 쌍(Dunn, bonferroni<0.05): [(0, 1), (0, 3), (0, 4), (0, 5), (0, 6), (2, 3), (2, 4)]
VIS_TYPE 테마시설(놀이공원, 워터파크): H=18.49, p=0.0051, p_fdr=0.0122 (FDR 유의)
    유의한 클러스터 쌍(Dunn, bonferroni<0.05): 없음
VIS_TYPE 상점: H=15.12, p=0.0194, p_fdr=0.0387 (FDR 유의)
    유의한 클러스

## SECTION 5. 미션 달성도

In [10]:
print("=" * 70)
print("[SECTION 5] 미션 달성도 - 동기 유형별 차이 검정")
print("=" * 70)


def parse_codes(s):
    if pd.isna(s):
        return set()
    return set(int(x) for x in str(s).split(";"))


travel["MISSION_SET"] = travel["TRAVEL_MISSION"].apply(parse_codes)
travel["CHECK_SET"] = travel["TRAVEL_MISSION_CHECK"].apply(parse_codes)
travel["ACHIEVE_RATE"] = [
    len(m & c) / len(m) if len(m) > 0 else np.nan
    for m, c in zip(travel["MISSION_SET"], travel["CHECK_SET"])
]
travel["DISCOVER_RATE"] = [
    len(c - m) / len(c) if len(c) > 0 else np.nan
    for m, c in zip(travel["MISSION_SET"], travel["CHECK_SET"])
]
print(f"ACHIEVE_RATE 계산 불가(미션 0건): {travel['ACHIEVE_RATE'].isna().sum()}건")
print(f"DISCOVER_RATE 계산 불가(체크 0건): {travel['DISCOVER_RATE'].isna().sum()}건")

travel_m = travel.merge(master[["TRAVELER_ID", "MOTIVE_CLUSTER"]], on="TRAVELER_ID", how="left")
print(travel_m.groupby("MOTIVE_CLUSTER")[["ACHIEVE_RATE", "DISCOVER_RATE"]].mean())

for col in ["ACHIEVE_RATE", "DISCOVER_RATE"]:
    sub = travel_m.dropna(subset=[col, "MOTIVE_CLUSTER"])
    groups = [g[col].values for _, g in sub.groupby("MOTIVE_CLUSTER")]
    stat, p = kruskal(*groups)
    print(f"{col} 클러스터간 차이 Kruskal-Wallis: H={stat:.2f}, p={p:.4f}")
    if p < 0.05:
        dunn = sp.posthoc_dunn(sub, val_col=col, group_col="MOTIVE_CLUSTER", p_adjust="bonferroni")
        sig_pairs = [(a, b) for a in dunn.index for b in dunn.columns
                     if a < b and dunn.loc[a, b] < 0.05]
        print(f"  유의한 클러스터 쌍(Dunn): {sig_pairs if sig_pairs else '없음'}")


[SECTION 5] 미션 달성도 - 동기 유형별 차이 검정
ACHIEVE_RATE 계산 불가(미션 0건): 0건
DISCOVER_RATE 계산 불가(체크 0건): 0건
                ACHIEVE_RATE  DISCOVER_RATE
MOTIVE_CLUSTER                             
0                   0.601986       0.351159
1                   0.607795       0.324192
2                   0.645868       0.300654
3                   0.602996       0.333333
4                   0.538818       0.333333
5                   0.650000       0.306122
6                   0.625992       0.286787
ACHIEVE_RATE 클러스터간 차이 Kruskal-Wallis: H=16.21, p=0.0127
  유의한 클러스터 쌍(Dunn): [(1, 4), (2, 4), (4, 6)]
DISCOVER_RATE 클러스터간 차이 Kruskal-Wallis: H=7.23, p=0.3002
